<a href="https://colab.research.google.com/github/MKMLLER/AllCups_AvitoPrice/blob/main/AllCups_3_MeanEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Импортируем библиотеки

In [ ]:
!pip install --upgrade huggingface_hub transformers

In [ ]:
!pip install torch torchvision catboost transformers -q

In [ ]:
import os
import random
import cv2
import re
import requests
from tqdm import tqdm

from PIL import Image
from IPython.display import display, Image
from PIL import Image as PILImage

import pandas as pd
import numpy as np
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
from torch import Tensor
from torch.optim import Adam

from transformers import AutoModel, AutoProcessor, AutoImageProcessor
from catboost import CatBoostRegressor, Pool

from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms
from torchvision.models.feature_extraction import create_feature_extractor

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_percentage_error as mape



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/train_images.zip /content/train_images.zip
!unzip -q /content/train_images.zip -d /content/train_images_unzip

In [ ]:
!cp /content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/test_images.zip /content/test_images.zip
!unzip -q /content/test_images.zip -d /content/test_images_unzip

# Получение эмбеддингов
- Рекомендую скачать папки с изображениями на свой google drive, чтобы не скачивать их с облака при каждом перезапуске сессии  

In [ ]:
train_data = pd.read_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/train_dataset.parquet') # Табличные данные
test_data = pd.read_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/test_dataset.parquet')

In [ ]:
class DataSet(Dataset): # Класс датасета, по индексу возращает преобразованную соответстующими трансформациями картинку
  def __init__(self, df, path, transform=None):
    self.df = df
    self.path = path
    self.transform = transform

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
        image_name = self.df.iloc[idx]['ID']

        # Создаем тензор сразу нужного размера (4 картинки, 3 канала, 224x224)
        final_images = torch.zeros(4, 3, 224, 224, dtype=torch.float32)
        mask = torch.zeros(4, dtype=torch.float32)

        for i in range(4):
            full_path = os.path.join(self.path, f'{image_name}_{i}.jpg')

            if os.path.exists(full_path):
                image = PILImage.open(full_path)

                if self.transform:
                    # Трансформируем картинку и СРАЗУ кладем в нужный индекс
                    final_images[i] = self.transform(image)

                mask[i] = 1.0

        return {
            'images': final_images,
            'mask': mask
        }

In [ ]:
class cvModel(nn.Module):
  def __init__(self, model_path='content/dinov2-base'):
    super().__init__()

    self.backbone = AutoModel.from_pretrained(model_path)

  def forward(self, x):

    batch_size = x['images'].size(0)

    x_images = x['images'].view(-1, 3, 224, 224)
    x_mask = x['mask'].view(-1)

    valid_indices = torch.nonzero(x_mask).squeeze(1)
    outputs = torch.zeros(x_images.size(0), 768, device=x_images.device, dtype=x_images.dtype)

    valid_images = x_images[valid_indices]
    backbone_outputs = self.backbone(pixel_values=valid_images).pooler_output

    outputs[valid_indices] = backbone_outputs
    outputs = outputs.view(batch_size, 4, 768)

    mask = x['mask'].unsqueeze(-1)


    outputs_sum = torch.sum(outputs * mask, dim=1)

    mask_sum = torch.sum(mask, dim=1).clamp(min=1.0)

    return outputs_sum / mask_sum

In [ ]:
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Устанавливаем git-lfs, если его нет
!git lfs install

# Клонируем только нужную модель (без истории коммитов, чтобы быстрее)
!git clone --depth 1 https://huggingface.co/facebook/dinov2-base content/dinov2-base

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
#processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
model = cvModel()
device = 'cuda'
model = model.to(device)

#train/test_images - папка на вашем диске
#train_dataset = DataSet(train_data, '/content/train_images_unzip', img_transform )
test_dataset = DataSet(test_data, '/content/test_images_unzip', img_transform )

'''train_loader = DataLoader(train_dataset,
    batch_size=64,          # Уменьшаем до 32, чтобы память не забивалась
    shuffle=False,
    num_workers=3,          # Убираем параллельные процессы CPU (убирает оверхед копирования)
    pin_memory=True)'''
test_loader = DataLoader(test_dataset,
                           batch_size=64,          # Уменьшаем до 32, чтобы память не забивалась
                            shuffle=False,
                            num_workers=0,          # Убираем параллельные процессы CPU (убирает оверхед копирования)
                            pin_memory=True)

In [ ]:
def get_embeddings(loader, model=model): # Достаем cls токены в качестве эмбеддингов картинок
  cls_tokens = []
  model.eval()
  with torch.no_grad():

    for batch in tqdm(loader):
      batch = {k : v.to(device) for k, v in batch.items()}


      outputs = model(batch)
      cls_tokens.append(outputs.cpu().numpy())

  return np.concatenate(cls_tokens, axis=0)



In [ ]:
#train_embeddings = get_embeddings(train_loader)
test_embeddings = get_embeddings(test_loader)

In [ ]:
#train_emb_df = pd.DataFrame(train_embeddings)
test_emb_df = pd.DataFrame(test_embeddings)
display(test_emb_df.shape)
#display(test_emb_df.shape)

In [ ]:
#train_emb_df.to_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/train_embeddings.parquet')
test_emb_df.to_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/test_embeddings.parquet')

# Обучение полносвязной сети на предсказание цены

In [ ]:
train_emb_df = pd.read_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/train_embeddings.parquet')
test_emb_df = pd.read_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/test_embeddings.parquet')
train_emb = train_emb_df.to_numpy()
test_emb = test_emb_df.to_numpy()


In [ ]:
# Функция для корректной работы MultiLabelBinarizer
def put_zero(df):

  if len(df) == 0:
    return '0'

  if len(df) == 1 and (df[0] == None or str(df[0]).strip().lower() == 'none'):
    return '0'

  return df

# Раскрываем фичи вида ['1', '3', '2'] в отдельные
def add_features(df, mlb):
  mul_columns = train_data.filter(regex='mul')

  for column in mul_columns:
    df[column] = df[column].apply(put_zero)

  features = []
  for column in mul_columns:
    features.append(mlb.fit_transform(df[column])[:, 1:])
    df = df.drop([column], axis=1)

  features_concat = pd.DataFrame(np.concat(features, axis=1))
  out = pd.concat([df, features_concat], axis=1)

  return out


In [ ]:
class PricePredictor(nn.Module):
    def __init__(self):
        super(PricePredictor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)



In [ ]:
mlb = MultiLabelBinarizer()

train = add_features(train_data, mlb)
test = add_features(test_data, mlb)

y = np.log1p(train.pop('price_TARGET')) # Берем логарифм от целевой переменной чтобы уменьшить разброс и облегчить обучение

cat_features = list(train.select_dtypes(include='object').columns) # Категориальные фичи
X = train.fillna(0)

In [ ]:
kfold = KFold(shuffle=False)

test_preds_folds = []
val_outputs = []

for i, (train_idx, test_idx) in enumerate(kfold.split(train_emb)):
    print(f"Split {i}")
    X_train, X_test, y_train, y_test = train_emb[train_idx], train_emb[test_idx], y.iloc[train_idx].values, y.iloc[test_idx].values

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)), batch_size=128, shuffle=True)

    model = PricePredictor().to('cuda')
    optim = Adam(model.parameters())
    criterion = nn.L1Loss()

    n_epochs = 20
    model.train()

    for epoch in range(n_epochs):
        for batch_x, batch_y in tqdm(train_loader):
          optim.zero_grad()
          batch_x = batch_x.to('cuda')
          batch_y = batch_y.to('cuda')

          preds = model(batch_x).squeeze()
          loss = criterion(preds, batch_y)
          loss.backward()

          optim.step()
        print(loss.item())

    model.eval()
    with torch.no_grad():
        X_val = torch.tensor(X_test, dtype=torch.float32).to('cuda')
        val_preds = (model(X_val).squeeze()).cpu().numpy()

        if 'pred_price' not in X.columns:
          X['pred_price'] = 0.0


        col_idx = X.columns.get_loc('pred_price')


        X.iloc[test_idx, col_idx] = np.expm1(val_preds)

    # На каждом шаге добавим предсказания моделью на валидации
        test_input = torch.tensor(test_emb).to('cuda')
        test_output = np.expm1(model(test_input).cpu().numpy())
        val_outputs.append(test_output)






In [ ]:
val_outputs_concat = np.concat(val_outputs, axis=1)
test_outputs_mean = np.mean(val_outputs_concat, axis=1)
test_outputs_mean


# Обучение catboost на предсказанных полносвязным слоем фичах

---



In [ ]:
X_train_catboost, X_test_catboost, y_train_catboost, y_test_catboost = train_test_split(X, y, test_size=0.2, random_state=42)

display(X_train_catboost['pred_price'])

train_pool = Pool(X_train_catboost, y_train_catboost, cat_features=cat_features)
test_pool = Pool(X_test_catboost, y_test_catboost, cat_features=cat_features)

catboost = CatBoostRegressor(10000,
                              learning_rate=0.15,
                              depth=6,
                              task_type='GPU',
                              loss_function='MAPE',
                              eval_metric='MAPE')

catboost.fit(train_pool,
              eval_set=test_pool,
              early_stopping_rounds=200,
              verbose=10)

In [ ]:

preds = np.expm1(catboost.predict(test_pool)) # Берем экспоненту от предсказаний, чтобы вернуть их в требуемый вид
print(f'Метрика MAPE: {mape(np.expm1(y_test), preds)}')

# Предсказание цен и отправка сабмита
 - Прогон тестовых эмбедингов через полносвязный слой, подача теста в катбуст

In [ ]:
# По первой оси склеиваем предсказания полносвязной модели и фичи для catboost
X_test = pd.concat([test.fillna(0), pd.DataFrame({ 'pred_price' : test_outputs_mean})], axis=1)

test_preds = catboost.predict(X_test)
test_preds_exp = np.expm1(test_preds)

In [ ]:
test_preds_exp.shape

In [ ]:
submission1 = pd.DataFrame(
    {
        'ID' : test['ID'],
        'target' : test_preds_exp
    }
)
submission1.to_csv('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/submission11.csv', index=False)

# Перебор параметров катбуста

In [ ]:
'''param_grid = {
    'depth': [6, 8, 12],
    'l2_leaf_reg': [3, 5, 7],
    'learning_rate': [0.03, 0.1, 0.2]
}

catboost2 = CatBoostRegressor(1500,
                              learning_rate=0.1,
                              task_type='GPU',
                              loss_function='MAPE',
                              eval_metric='MAPE',
                              cat_features=cat_features)

grid_search_result = catboost2.grid_search(
    param_grid,
    X=X_train_catboost,
    y=y_train_catboost,
    cv=2,
    partition_random_seed=42,
)'''